In [28]:
import pandas as pd
import re
import numpy as np
import os
from pathlib import Path
from underthesea import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT_DIR = Path(os.getcwd()).parent if "source" in os.getcwd() else Path(os.getcwd())
DATA_DIR = ROOT_DIR / "data"

teencode_dict = {
    "ks": "khách sạn", "ko": "không", "k": "không", "đc": "được",
    "nv": "nhân viên", "tsao": "tại sao", "vs": "với", "ph": "phòng",
    "view": "cảnh quan", "rooftoo": "sân thượng", "bthg": "bình thường"
}

In [2]:
def clean_vietnamese_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r'[\n\t]+', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    
    words = text.split()
    cleaned_words = [teencode_dict.get(word, word) for word in words]
    text = ' '.join(cleaned_words)
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    return word_tokenize(text, format="text")

def drop_duplicates_by_threshold(df, column='cleaned_comment', threshold=0.8):
    print(f"--- Bắt đầu lọc trùng lặp ngữ nghĩa (Threshold: {threshold}) ---")
    
    # Vector TF-IDF
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df[column].astype(str))
    
    # Tính toán Cosine Similarity
    cosine_sim = cosine_similarity(tfidf_matrix)
    
    to_drop = set()
    num_rows = cosine_sim.shape[0]
    for i in range(num_rows):
        if i in to_drop: continue
        similar_indices = np.where(cosine_sim[i] > threshold)[0]
        for idx in similar_indices:
            if idx > i: 
                to_drop.add(idx)
                
    df_result = df.iloc[list(set(range(num_rows)) - to_drop)].reset_index(drop=True)
    print(f"Đã loại bỏ {len(to_drop)} dòng tương đồng vượt ngưỡng.")
    return df_result

In [3]:
input_path = DATA_DIR / "crawl_test.csv"
output_path = DATA_DIR / "crawl_test_cleaned.csv"

if input_path.exists():
    df = pd.read_csv(input_path)
    print(f"Số lượng raw data: {len(df)}")

    df = df.dropna(subset=['comment'])

    print(" Đang chuẩn hóa và tách từ...")
    df['cleaned_comment'] = df['comment'].apply(clean_vietnamese_text)
    df = df[df['cleaned_comment'] != ""] 

    df = drop_duplicates_by_threshold(df, threshold=0.8)

    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"Đã lưu {len(df)} dòng dữ liệu sạch tại: {output_path}")
    
    display(df[['comment', 'cleaned_comment']].head())
else:
    print(f"Không tìm thấy file tại: {input_path}")

Số lượng raw data: 519
 Đang chuẩn hóa và tách từ...
--- Bắt đầu lọc trùng lặp ngữ nghĩa (Threshold: 0.8) ---
Đã loại bỏ 3 dòng tương đồng vượt ngưỡng.
Đã lưu 516 dòng dữ liệu sạch tại: d:\Year 3\HK2\ML\lab1\data\crawl_test_cleaned.csv


,comment,cleaned_comment
0,Ở đây cũng tạm ổn. Lần sau lại tiếp,ở đây cũng tạm ổn lần sau lại tiếp
1,phòng rất sạch và thơm nhưng có góp ý nhỏ là đ...,phòng rất sạch và thơm nhưng có góp_ý nhỏ là đ...
2,Ở đầu giường không có chỗ sạc pin điện thoại,ở đầu giường không có chỗ sạc pin điện_thoại
3,mình rất hài lòng với không khí như gia đình ở...,mình rất hài_lòng với không_khí như gia_đình ở...
4,"khách sạn sạch, đồ ăn ngon, không gia yên tĩnh",khách_sạn sạch đồ_ăn ngon không gia_yên_tĩnh


In [5]:
import json
import uuid

def format_label_studio_data(input_json_name, output_json_name):
    print("---CHUYỂN ĐỔI JSON TỪ LABEL STUDIO SANG KHUNG DỮ LIỆU ---")
    
    input_path = DATA_DIR / input_json_name
    output_path = DATA_DIR / output_json_name
    
    if not input_path.exists():
        print(f"Không tìm thấy file tại: {input_path}")
        return

    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
            
        category_map = {
            "Room": "Room_Facilities",
            "Service": "Service_Staff",
            "Location": "Location",
            "Food": "Food_Beverage",
            "Price": "Price_Value",
            "General": "General"
        }
        
        formatted_data = []
        
        for item in raw_data:
            text = item.get("cleaned_comment", "")
            if not text:
                continue
                
            entry = {
                "id": str(uuid.uuid4()), 
                "text": text,
                "aspects": []
            }
            
            for ls_key, standard_category in category_map.items():
                if ls_key in item and item[ls_key]:
                    entry["aspects"].append({
                        "category": standard_category,
                        "sentiment": item[ls_key]
                    })
            
            if entry["aspects"]:
                formatted_data.append(entry)

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(formatted_data, f, ensure_ascii=False, indent=4)
            
        print(f"Đã chuyển đổi {len(formatted_data)} câu có nhãn.")
        print(f"File chuẩn lưu tại: {output_path}")
        
    except Exception as e:
        print(f"Lỗi trong quá trình format: {e}")

format_label_studio_data('label_studio.json', 'label_studio_formated.json')

---CHUYỂN ĐỔI JSON TỪ LABEL STUDIO SANG KHUNG DỮ LIỆU ---
Đã chuyển đổi 1009 câu có nhãn.
File chuẩn lưu tại: d:\Year 3\HK2\ML\lab1\data\label_studio_formated.json


In [ ]:
import pandas as pd
import json
from pathlib import Path

def json_to_sentiment_matrix(input_json_name, output_csv_name):
    print("--- BẮT ĐẦU TẠO MA TRẬN DỮ LIỆU ---")
    
    input_path = DATA_DIR / input_json_name
    output_path = DATA_DIR / output_csv_name
    
    if not input_path.exists():
        print(f"Không tìm thấy file: {input_path}")
        return

    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Negative: 1, None : 0, Positive: 2
    sentiment_mapping = {"Positive": 2, "Negative": 1}
    categories = ["Room_Facilities", "Service_Staff", "Location", "Food_Beverage", "Price_Value", "General"]
    
    flattened_data = []
    
    for item in data:
        row = {
            "id": item.get("id", ""), 
            "text": item.get("text", "")
        }
        for cat in categories:
            row[cat] = 0 
        
        for aspect in item.get("aspects", []):
            cat = aspect.get("category")
            sent = aspect.get("sentiment")
            if cat in categories and sent in sentiment_mapping:
                row[cat] = sentiment_mapping[sent]
        
        flattened_data.append(row)
        
    df_matrix = pd.DataFrame(flattened_data)
    
    df_matrix = df_matrix[df_matrix[categories].abs().sum(axis=1) > 0]
    
    df_matrix.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    print(f"Đã tạo ma trận với {len(df_matrix)} mẫu chất lượng.")
    print(f"File lưu tại: {output_path}")
    
    return df_matrix

df_final_matrix = json_to_sentiment_matrix('dataset.json', 'dataset_matrix.csv')

--- BẮT ĐẦU TẠO MA TRẬN DỮ LIỆU ---
Đã tạo ma trận với 9690 mẫu chất lượng.
File lưu tại: d:\Year 3\HK2\ML\lab1\data\dataset_matrix.csv


In [31]:
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def run_full_eda_report(json_path, csv_matrix_path):
    print("="*60)
    print(" BÁO CÁO THỐNG KÊ EDA CHI TIẾT DÀNH CHO BẢN WORD")
    print("="*60)
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df_text = pd.DataFrame(data)
    
    df_matrix = pd.read_csv(csv_matrix_path)
    
    # PHẦN 1: THỐNG KÊ MÔ TẢ & VĂN BẢN
    print("\n1. THỐNG KÊ MÔ TẢ (DỮ LIỆU PHI CẤU TRÚC ):")
    df_text['word_count'] = df_text['text'].apply(lambda x: len(str(x).split()))
    
    total_samples = len(df_text)
    mean_len = df_text['word_count'].mean()
    median_len = df_text['word_count'].median()
    min_len = df_text['word_count'].min()
    max_len = df_text['word_count'].max()
    
    q1 = df_text['word_count'].quantile(0.25)
    q3 = df_text['word_count'].quantile(0.75)
    iqr = q3 - q1
    variance_len = df_text['word_count'].var()
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers_count = df_text[(df_text['word_count'] < lower_bound) | (df_text['word_count'] > upper_bound)].shape[0]
    
    duplicates_count = df_text['text'].duplicated().sum()
    
    def count_special_chars(text):
        special_chars = re.sub(r'[\w\s]', '', str(text))
        return len(special_chars)
        
    df_text['special_char_count'] = df_text['text'].apply(count_special_chars)
    df_text['char_count'] = df_text['text'].apply(lambda x: len(str(x)))
    special_char_ratio = (df_text['special_char_count'].sum() / df_text['char_count'].sum()) * 100

    print(f"- Số lượng mẫu (Total samples): {total_samples} câu")
    print(f"- Trung bình (Mean): {mean_len:.2f} từ/câu")
    print(f"- Trung vị (Median): {median_len:.2f} từ/câu")
    print(f"- Min / Max: {min_len} / {max_len} từ/câu")
    print(f"- Khoảng tứ phân vị (IQR): {iqr:.2f}")
    print(f"- Phương sai (Variance): {variance_len:.2f}")
    print(f"- Tỉ lệ ký tự đặc biệt/nhiễu: {special_char_ratio:.2f}% trên tổng số ký tự")
    print(f"- Số lượng bình luận ngoại lệ (Outliers): {outliers_count} câu")
    print(f"- Dữ liệu trùng lặp (Duplicates): {duplicates_count} câu")
    print(f"- Tỉ lệ dữ liệu thiếu (Missing values): 0%")

    # PHẦN 2: PHÂN TÍCH PHÂN BỐ NHÃN (0: None, 1 : Negative, 2 : Positive)
    print("\n2. PHÂN TÍCH PHÂN BỐ NHÃN:")
    categories = ["Room_Facilities", "Service_Staff", "Location", "Food_Beverage", "Price_Value", "General"]
    
    for cat in categories:
        valid_rows = df_matrix[df_matrix[cat] != 0][cat]
        total_valid = len(valid_rows)
        
        if total_valid > 0:
            pos_count = (valid_rows == 2).sum()
            neg_count = (valid_rows == 1).sum()
            pos_ratio = (pos_count / total_valid) * 100
            neg_ratio = (neg_count / total_valid) * 100
            
            print(f"  * {cat.upper()}: Tổng có {total_valid} đánh giá.")
            print(f"    -> Positive: {pos_count} ({pos_ratio:.2f}%) | Negative: {neg_count} ({neg_ratio:.2f}%)")
        else:
            print(f"  * {cat.upper()}: Không có dữ liệu.")


if __name__ == "__main__":
    
    input_json_name = 'dataset.json'
    input_csv_name = 'dataset_matrix.csv'
    
    json_input = DATA_DIR / input_json_name
    csv_matrix_input = DATA_DIR / input_csv_name
    
    if not json_input.exists() or not csv_matrix_input.exists():
        print(f"Lỗi: Không tìm thấy file dữ liệu tại: {DATA_DIR}")
    else:
        run_full_eda_report(str(json_input), str(csv_matrix_input))

 BÁO CÁO THỐNG KÊ EDA CHI TIẾT DÀNH CHO BẢN WORD

1. THỐNG KÊ MÔ TẢ (DỮ LIỆU PHI CẤU TRÚC ):
- Số lượng mẫu (Total samples): 9695 câu
- Trung bình (Mean): 24.34 từ/câu
- Trung vị (Median): 15.00 từ/câu
- Min / Max: 1 / 597 từ/câu
- Khoảng tứ phân vị (IQR): 18.00
- Phương sai (Variance): 858.16
- Tỉ lệ ký tự đặc biệt/nhiễu: 0.00% trên tổng số ký tự
- Số lượng bình luận ngoại lệ (Outliers): 839 câu
- Dữ liệu trùng lặp (Duplicates): 0 câu
- Tỉ lệ dữ liệu thiếu (Missing values): 0%

2. PHÂN TÍCH PHÂN BỐ NHÃN:
  * ROOM_FACILITIES: Tổng có 5145 đánh giá.
    -> Positive: 2731 (53.08%) | Negative: 2414 (46.92%)
  * SERVICE_STAFF: Tổng có 3640 đánh giá.
    -> Positive: 2780 (76.37%) | Negative: 860 (23.63%)
  * LOCATION: Tổng có 1980 đánh giá.
    -> Positive: 1754 (88.59%) | Negative: 226 (11.41%)
  * FOOD_BEVERAGE: Tổng có 1811 đánh giá.
    -> Positive: 1118 (61.73%) | Negative: 693 (38.27%)
  * PRICE_VALUE: Tổng có 1441 đánh giá.
    -> Positive: 915 (63.50%) | Negative: 526 (36.50%)
  * 

In [32]:
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

def generate_eda_charts(input_json):
    print("--- BẮT ĐẦU VẼ BIỂU ĐỒ BÁO CÁO (EDA) ---")
    
    # 1. Xử lý đường dẫn chuẩn xác
    try:
        current_dir = Path(__file__).resolve().parent
    except NameError:
        current_dir = Path(os.getcwd())
    
    # Đảm bảo lưu vào LAB1/data/eda_charts
    root_path = current_dir.parent if current_dir.name == "source" else current_dir
    output_dir = root_path / 'data' / 'eda_charts'
    output_dir.mkdir(parents=True, exist_ok=True)

    # 2. Kiểm tra Font tiếng Việt (Bắt buộc)
    font_path = "C:/Windows/Fonts/arial.ttf" # Đường dẫn mặc định trên Windows
    if not os.path.exists(font_path):
        font_path = None # Nếu không tìm thấy sẽ dùng font mặc định (nhưng dễ lỗi dấu)

    if not os.path.exists(input_json):
        print(f"Không tìm thấy file: {input_json}")
        return

    with open(input_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    aspects_list = []
    text_data = [] 
    categories = ["Room_Facilities", "Service_Staff", "Location", "Food_Beverage", "Price_Value", "General"]
    sentiment_map = {"Positive": 2, "Negative": 1}
    matrix_rows = []

    for item in data:
        text = item.get('text', '')
        row = {cat: 0 for cat in categories}
        sentiments_in_item = []
        
        for aspect in item.get('aspects', []):
            cat = aspect['category']
            sent = aspect['sentiment']
            aspects_list.append({'Category': cat, 'Sentiment': sent})
            sentiments_in_item.append(sent)
            if cat in categories:
                row[cat] = sentiment_map.get(sent, 0)
        
        # Lưu text kèm theo nhãn tổng quát để vẽ WordCloud không bị lặp
        if "Positive" in sentiments_in_item:
            text_data.append({'text': text, 'Sentiment': 'Positive'})
        if "Negative" in sentiments_in_item:
            text_data.append({'text': text, 'Sentiment': 'Negative'})
            
        matrix_rows.append(row)
            
    df_aspects = pd.DataFrame(aspects_list)
    df_text = pd.DataFrame(text_data)
    df_matrix = pd.DataFrame(matrix_rows)

    # ==========================================
    # ẢNH 1: PHÂN PHỐI (Sửa lỗi palette)
    # ==========================================
    print("1. Đang vẽ ảnh phân phối Cảm xúc & Khía cạnh...")
    fig1, axes1 = plt.subplots(1, 2, figsize=(20, 8))
    
    sentiment_palette = {'Positive': '#5cb85c', 'Negative': '#d9534f'}
    sns.countplot(data=df_aspects, x='Sentiment', ax=axes1[0], hue='Sentiment', 
                  palette=sentiment_palette, order=['Positive', 'Negative'], legend=False)
    axes1[0].set_title('Phân phối Cảm xúc (Sentiment)', fontsize=16, fontweight='bold')
    
    category_order = df_aspects['Category'].value_counts().index
    sns.countplot(data=df_aspects, y='Category', ax=axes1[1], hue='Category',
                  order=category_order, palette='viridis', legend=False)
    axes1[1].set_title('Phân phối Khía cạnh (Aspect Category)', fontsize=16, fontweight='bold')
    # Thêm số lượng cho cột đứng (get_height)
    for p in axes1[0].patches:
        axes1[0].annotate(f'{int(p.get_height())}', 
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='bottom', fontsize=14, 
                      xytext=(0, 5), textcoords='offset points')

# --- Biểu đồ 2: Category (Cột ngang) ---
    category_order = df_aspects['Category'].value_counts().index
    sns.countplot(data=df_aspects, y='Category', ax=axes1[1], hue='Category',
              order=category_order, palette='viridis', legend=False)
    axes1[1].set_title('Phân phối Khía cạnh (Aspect Category)', fontsize=16, fontweight='bold')

# CÁCH THÊM SỐ CHO BIỂU ĐỒ NGANG (get_width)
    for p in axes1[1].patches:
        width = p.get_width()
        axes1[1].annotate(f'{int(width)}', 
                      (width, p.get_y() + p.get_height() / 2.),
                      ha='left', va='center', fontsize=12, 
                      xytext=(5, 0), textcoords='offset points')
    plt.tight_layout()
    plt.savefig(output_dir / '1_Distribution_Charts.png', dpi=300)
    plt.close()

    # ==========================================
    # ẢNH 2: WORD CLOUDS (Sửa lỗi trắng ảnh)
    # ==========================================
    print("2. Đang vẽ ảnh Word Clouds ghép...")
    text_pos = " ".join(df_text[df_text['Sentiment'] == 'Positive']['text'].astype(str)).strip()
    text_neg = " ".join(df_text[df_text['Sentiment'] == 'Negative']['text'].astype(str)).strip()
    
    stopwords_vn = set(["có", "không", "là", "và", "thì", "mà", "ở", "của", "cho", "để", "với", "nhưng", "rất", "cũng", "được", "này", "khi", "trong", "khách", "sạn"])

    fig2, axes2 = plt.subplots(1, 2, figsize=(24, 10))

    if text_pos:
        wc_pos = WordCloud(width=800, height=600, background_color='white', colormap='Greens',
                           font_path=font_path, stopwords=stopwords_vn, max_words=150).generate(text_pos)
        axes2[0].imshow(wc_pos, interpolation='bilinear')
        axes2[0].set_title('Từ khóa POSITIVE', fontsize=20, color='green', fontweight='bold')
    else:
        axes2[0].text(0.5, 0.5, 'KHÔNG CÓ DỮ LIỆU POSITIVE', ha='center')
    axes2[0].axis('off')

    if text_neg:
        wc_neg = WordCloud(width=800, height=600, background_color='black', colormap='Reds',
                           font_path=font_path, stopwords=stopwords_vn, max_words=150).generate(text_neg)
        axes2[1].imshow(wc_neg, interpolation='bilinear')
        axes2[1].set_title('Từ khóa NEGATIVE', fontsize=20, color='red', fontweight='bold')
    else:
        axes2[1].text(0.5, 0.5, 'KHÔNG CÓ DỮ LIỆU NEGATIVE', ha='center', color='white')
    axes2[1].axis('off')
    
    plt.savefig(output_dir / '2_Combined_WordClouds.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # ẢNH 3: MA TRẬN TƯƠNG QUAN (HEATMAP)
    # ==========================================
    print("3. Đang vẽ Ma trận tương quan (Heatmap)...")
    df_corr = df_matrix[categories].replace(0, np.nan)
    corr_matrix = df_corr.corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, center=0)
    plt.title('Ma trận Tương quan giữa 6 Khía cạnh\n(Dựa trên cảm xúc thực tế)', fontsize=16, fontweight='bold')
    
    plt.savefig(os.path.join(output_dir, '3_Correlation_Heatmap.png'), dpi=300, bbox_inches='tight')
    plt.close()

    
    print("4. Đang vẽ biểu đồ phân bố độ dài câu...")
    
    if 'word_count' not in df_text.columns:
        df_text['word_count'] = df_text['text'].apply(lambda x: len(str(x).split()))

    plt.figure(figsize=(10, 6))
    
    sns.histplot(df_text['word_count'], 
                 bins=50, 
                 kde=True, 
                 color='skyblue', 
                 edgecolor='black',
                 line_kws={'linewidth': 2})

    plt.title('Phân bố độ dài câu bình luận (Số lượng từ)', fontsize=14, fontweight='bold')
    plt.xlabel('Số lượng từ trong 1 câu', fontsize=12)
    plt.ylabel('Số lượng câu', fontsize=12)
    
    plt.grid(axis='y', linestyle='--', alpha=0.6)

    output_path = os.path.join(output_dir, '4_Length_Distribution.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

if __name__ == "__main__":
    DATA_DIR = ROOT_DIR / "data"
    input_file = DATA_DIR / 'dataset.json'
    
    if input_file.exists():
        print(f"Đang xử lý file: {input_file}")
        generate_eda_charts(str(input_file))
    else:
        print(f"Lỗi: Không tìm thấy file tại {input_file}")

Đang xử lý file: d:\Year 3\HK2\ML\lab1\data\dataset.json
--- BẮT ĐẦU VẼ BIỂU ĐỒ BÁO CÁO (EDA) ---
1. Đang vẽ ảnh phân phối Cảm xúc & Khía cạnh...
2. Đang vẽ ảnh Word Clouds ghép...
3. Đang vẽ Ma trận tương quan (Heatmap)...
4. Đang vẽ biểu đồ phân bố độ dài câu...
